# 04 — SQL Analytics with DuckDB

This notebook uses DuckDB to query the FIPE Silver and Gold Parquet datasets with SQL.

In [1]:
from pathlib import Path

from fipe_pipeline.duckdb_layer import (
    connect_duckdb,
    register_parquet_views,
    validate_duckdb_layer,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

In [2]:
con = connect_duckdb()
register_parquet_views(con)
validate_duckdb_layer(con)

DuckDBValidationResult(silver_rows=9528944, gold_rows=9528944, row_counts_match=True, silver_first_period=(2001, 1), silver_last_period=(2026, 9), gold_first_period=(2001, 1), gold_last_period=(2026, 9), periods_match=True)

## 1. Dataset overview

In [3]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT codigo_fipe) AS distinct_fipe_codes,
    MIN(data_referencia) AS first_period,
    MAX(data_referencia) AS last_period
FROM gold_fipe
""").df()

,rows,distinct_fipe_codes,first_period,last_period
0,9528944,11398,2001-01-01,2026-09-01


## 2. Monthly record volume

In [4]:
monthly_volume = con.sql("""
SELECT
    data_referencia,
    COUNT(*) AS rows
FROM gold_fipe
GROUP BY data_referencia
ORDER BY data_referencia
""").df()

monthly_volume.tail(12)

,data_referencia,rows
297,2025-10-01,49174
298,2025-11-01,49371
299,2025-12-01,49524
300,2026-01-01,49676
301,2026-02-01,49773
302,2026-03-01,49987
303,2026-04-01,50129
304,2026-05-01,50252
305,2026-06-01,50395
306,2026-07-01,50599


## 3. Median vehicle price by month

In [5]:
monthly_median_price = con.sql("""
SELECT
    data_referencia,
    MEDIAN(valor_centavos) / 100.0 AS median_price_brl
FROM gold_fipe
GROUP BY data_referencia
ORDER BY data_referencia
""").df()

monthly_median_price.tail(12)

,data_referencia,median_price_brl
297,2025-10-01,55777.0
298,2025-11-01,56030.0
299,2025-12-01,56268.5
300,2026-01-01,56536.5
301,2026-02-01,56499.0
302,2026-03-01,56632.0
303,2026-04-01,56840.0
304,2026-05-01,57014.5
305,2026-06-01,57258.0
306,2026-07-01,57356.0


## 4. Vehicle distribution by type

In [6]:
con.sql("""
SELECT
    tipo_veiculo,
    COUNT(*) AS rows,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM gold_fipe
GROUP BY tipo_veiculo
ORDER BY rows DESC
""").df()

,tipo_veiculo,rows,pct
0,carro,5862186,61.52
1,caminhão,2166427,22.74
2,moto,1500331,15.74


## 5. Top brands by current median price

In [7]:
con.sql("""
WITH latest_period AS (
    SELECT MAX(data_referencia) AS max_date
    FROM gold_fipe
)
SELECT
    nome_marca,
    COUNT(*) AS rows,
    MEDIAN(valor_centavos) / 100.0 AS median_price_brl
FROM gold_fipe
WHERE data_referencia = (SELECT max_date FROM latest_period)
GROUP BY nome_marca
HAVING COUNT(*) >= 10
ORDER BY median_price_brl DESC
LIMIT 10
""").df()

,nome_marca,rows,median_price_brl
0,LAMBORGHINI,87,3693179.0
1,Rolls-Royce,29,3186329.0
2,Mclaren,49,2813000.0
3,Ferrari,209,2138539.0
4,ASTON MARTIN,52,2115363.0
5,Porsche,918,567979.5
6,DAF,445,531099.0
7,ZEEKR,11,411855.0
8,Maserati,132,378300.0
9,BEPOBUS,21,329869.0


## 6. Fuel mix in the latest month

In [8]:
con.sql("""
WITH latest_period AS (
    SELECT MAX(data_referencia) AS max_date
    FROM gold_fipe
)
SELECT
    nome_combustivel,
    COUNT(*) AS rows,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM gold_fipe
WHERE data_referencia = (SELECT max_date FROM latest_period)
GROUP BY nome_combustivel
ORDER BY rows DESC
""").df()

,nome_combustivel,rows,pct
0,Gasolina,24633,48.29
1,Diesel,16608,32.56
2,Flex,7150,14.02
3,Híbrido,1112,2.18
4,Elétrico,959,1.88
5,Álcool,466,0.91
6,Gás Natural,84,0.16


## 7. Example Silver vs Gold reconciliation in SQL

In [9]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM silver_fipe) AS silver_rows,
    (SELECT COUNT(*) FROM gold_fipe) AS gold_rows,
    (SELECT COUNT(*) FROM silver_fipe)
        = (SELECT COUNT(*) FROM gold_fipe) AS row_counts_match
""").df()

,silver_rows,gold_rows,row_counts_match
0,9528944,9528944,True


In [11]:
from fipe_pipeline.analytics_views import (
    create_analytics_views,
)

create_analytics_views(con)

In [12]:
con.sql("""
SELECT *
FROM vw_monthly_market_summary
ORDER BY data_referencia DESC
LIMIT 12
""").df()

,data_referencia,ano_referencia,mes_referencia,rows,distinct_fipe_codes,median_price_brl
0,2026-09-01,2026,9,51012,11396,57724.5
1,2026-08-01,2026,8,50838,11357,57725.0
2,2026-07-01,2026,7,50599,11289,57356.0
3,2026-06-01,2026,6,50395,11238,57258.0
4,2026-05-01,2026,5,50252,11210,57014.5
5,2026-04-01,2026,4,50129,11174,56840.0
6,2026-03-01,2026,3,49987,11135,56632.0
7,2026-02-01,2026,2,49773,11105,56499.0
8,2026-01-01,2026,1,49676,11082,56536.5
9,2025-12-01,2025,12,49524,11049,56268.5


In [13]:
con.sql("""
SELECT *
FROM vw_latest_brand_summary
LIMIT 10
""").df()

,nome_marca,rows,distinct_fipe_codes,median_price_brl,min_price_brl,max_price_brl
0,LAMBORGHINI,87,28,3693179.0,1032542.0,9749142.0
1,Rolls-Royce,29,8,3186329.0,1822082.0,6842472.0
2,Mclaren,49,16,2813000.0,1569889.0,5269692.0
3,Ferrari,209,59,2138539.0,129226.0,8266065.0
4,ASTON MARTIN,52,23,2115363.0,461739.0,6000000.0
5,ARROW,5,1,587795.0,524501.0,689000.0
6,Porsche,918,185,567979.5,37750.0,4207319.0
7,DAF,445,123,531099.0,189226.0,1156666.0
8,ZEEKR,11,5,411855.0,234409.0,542000.0
9,Denza,5,2,405000.0,365120.0,440500.0


In [14]:
con.sql("""
SELECT *
FROM vw_latest_fuel_mix
""").df()

,nome_combustivel,sigla_combustivel,rows,pct,median_price_brl
0,Gasolina,g,24633,48.29,31002.0
1,Diesel,d,16608,32.56,133510.0
2,Flex,f,7150,14.02,52419.0
3,Híbrido,h,1112,2.18,319222.0
4,Elétrico,l,959,1.88,214900.0
5,Álcool,e,466,0.91,11239.0
6,Gás Natural,n,84,0.16,989203.5


## 8. Close connection

In [ ]:
con.close()